# LDTF-BERT: 3 thành viên train, 1 thành viên phân tích

Notebook dùng chung cho bốn thành viên:

- Member 1: train sáu model với seed `42`;
- Member 2: train sáu model với seed `1337`;
- Member 3: train sáu model với seed `2024`;
- Member 4: kiểm tra và tổng hợp kết quả validation của cả ba seed.

Ba trainer phải bật **Runtime > Change runtime type > T4 GPU**. Member 4 có thể dùng CPU. Notebook không đánh giá tập test.


## 0. Chuẩn bị thư mục Drive dùng chung

Cả bốn tài khoản cần có quyền chỉnh sửa một thư mục Drive dùng chung. Mỗi người thêm shortcut của thư mục đó vào `MyDrive` với tên `LDTF_4_MEMBERS`.

Chỉ cần tải hai file train/validation lên Drive theo cấu trúc:

```text
LDTF_4_MEMBERS/
  data/processed/
    research_train.parquet
    research_validation.parquet
```

Không đặt file test vào notebook làm việc nhóm. Test tiếp tục được khóa cho lần đánh giá cuối cùng riêng biệt.


## 1. Mỗi thành viên chỉ sửa ô này

In [ ]:
MEMBER_ID = 1  # Chọn đúng 1, 2, 3 hoặc 4

SHARED_DRIVE_ROOT = '/content/drive/MyDrive/LDTF_4_MEMBERS'
REPO_URL = 'https://github.com/thanh1912-ut/LDTF_bert.git'
REPO_BRANCH = 'data/chi-bao-merge-datasets'
PROJECT_DIR = '/content/LDTF_bert'

assert MEMBER_ID in (1, 2, 3, 4), 'MEMBER_ID phải là 1, 2, 3 hoặc 4'
print('Đã chọn MEMBER_ID =', MEMBER_ID)


## 2. Mount Drive và lấy source code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import sys
from pathlib import Path

SHARED_ROOT = Path(SHARED_DRIVE_ROOT)
PROJECT = Path(PROJECT_DIR)
assert SHARED_ROOT.is_dir(), (
    f'Không thấy {SHARED_ROOT}. Hãy thêm shortcut thư mục dùng chung vào MyDrive.'
)

if not PROJECT.exists():
    subprocess.run(
        ['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(PROJECT)],
        check=True,
    )
else:
    assert (PROJECT / '.git').is_dir(), f'{PROJECT} tồn tại nhưng không phải Git repo'
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True
).strip()
print('Project:', PROJECT)
print('Branch :', REPO_BRANCH)
print('Commit :', GIT_COMMIT)


## 3. Cài thư viện và đọc config chung

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')],
    check=True,
)

import hashlib
import json

CONFIG_PATH = PROJECT / 'configs' / 'colab_4_members.json'
TEAM_CONFIG = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
CONFIG_SHA256 = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
MEMBER = TEAM_CONFIG['members'][str(MEMBER_ID)]
RESULTS_ROOT = SHARED_ROOT / TEAM_CONFIG['storage']['results_subdir']
ANALYSIS_ROOT = SHARED_ROOT / TEAM_CONFIG['storage']['analysis_subdir']
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

print('Vai trò    :', MEMBER['role'])
print('Phân công  :', MEMBER['label'])
print('Config hash:', CONFIG_SHA256)
print('Kết quả tại:', RESULTS_ROOT)


## 4. Kiểm tra GPU và dữ liệu cho Member 1-3

In [ ]:
import shutil

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

if MEMBER['role'] == 'trainer':
    import torch
    assert torch.cuda.is_available(), 'Chưa bật GPU trong Colab Runtime'
    gpu_name = torch.cuda.get_device_name(0)
    assert 'T4' in gpu_name.upper(), f'Config nhóm yêu cầu T4, hiện tại là {gpu_name}'
    print('GPU:', gpu_name)

    dataset_cfg = TEAM_CONFIG['dataset']
    drive_data = SHARED_ROOT / dataset_cfg['drive_processed_subdir']
    local_root = Path(dataset_cfg['local_data_root'])
    local_processed = local_root / 'processed'
    local_processed.mkdir(parents=True, exist_ok=True)

    for filename in (dataset_cfg['train_file'], dataset_cfg['validation_file']):
        source = drive_data / filename
        destination = local_processed / filename
        assert source.is_file(), f'Thiếu dữ liệu trên Drive: {source}'
        expected_hash = dataset_cfg['sha256'][filename]
        source_hash = sha256_file(source)
        assert source_hash == expected_hash, f'Checksum sai: {source}'
        if not destination.exists() or sha256_file(destination) != expected_hash:
            print('Đang copy vào ổ cục bộ:', filename)
            shutil.copy2(source, destination)
        assert sha256_file(destination) == expected_hash

    os.environ['LDTF_DATA_DIR'] = str(local_root)

    import pandas as pd
    checks = (
        ('train', local_processed / dataset_cfg['train_file']),
        ('validation', local_processed / dataset_cfg['validation_file']),
    )
    for split, path in checks:
        labels = pd.read_parquet(path, columns=['label'])['label']
        counts = labels.value_counts().sort_index().tolist()
        assert len(labels) == dataset_cfg['expected_rows'][split]
        assert counts == dataset_cfg['expected_label_counts'][split]
        print(f'{split:<10}: {len(labels):,} dòng, nhãn {counts}')

    assert not (local_processed / dataset_cfg['sealed_test_file']).exists(), (
        'Notebook nhóm không được copy tập test vào vùng train'
    )
else:
    print('Member 4 không cần GPU hoặc dữ liệu gốc để tổng hợp validation.')


## 5. Smoke test tự động cho Member 1-3

In [ ]:
if MEMBER['role'] == 'trainer':
    marker_dir = RESULTS_ROOT / '_team_checks'
    marker_dir.mkdir(parents=True, exist_ok=True)
    smoke_marker = marker_dir / f'member_{MEMBER_ID}_{GIT_COMMIT}.smoke_passed'
    if smoke_marker.exists():
        print('Smoke test đã PASS cho member và commit này.')
    else:
        subprocess.run(
            [sys.executable, '-m', 'scripts.smoke_test', '--quick'],
            cwd=PROJECT,
            check=True,
        )
        smoke_marker.write_text('PASS\n', encoding='utf-8')
        print('Smoke test: PASS')
else:
    print('Bỏ qua smoke test vì Member 4 chỉ phân tích kết quả.')


## 6. Hàng đợi train cho Member 1-3

Mỗi trainer chạy cùng sáu model với seed được giao. Nếu Colab ngắt, mở lại notebook với cùng `MEMBER_ID` và chạy từ đầu; job chưa xong sẽ tự dùng `last.pt` để resume, job hoàn tất sẽ được bỏ qua.


In [ ]:
from datetime import datetime, timezone

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    temporary.replace(path)

if MEMBER['role'] == 'trainer':
    protocol = TEAM_CONFIG['protocol']
    seed = int(MEMBER['seed'])
    jobs = list(protocol['runs'])
    print(f'Member {MEMBER_ID}: {len(jobs)} job, seed {seed}')

    for position, run_name in enumerate(jobs, start=1):
        run_label = f'{run_name}_seed{seed}'
        output_dir = RESULTS_ROOT / run_label
        summary_path = output_dir / 'run_summary.json'
        last_path = output_dir / 'last.pt'
        metadata_path = output_dir / 'team_job_metadata.json'

        print(f'\n[{position}/{len(jobs)}] {run_label}')
        output_dir.mkdir(parents=True, exist_ok=True)
        resume = last_path.exists()
        previous_metadata = (
            json.loads(metadata_path.read_text(encoding='utf-8'))
            if metadata_path.exists() else {}
        )

        if summary_path.exists():
            existing = json.loads(summary_path.read_text(encoding='utf-8'))
            signature = existing.get('data_signature', {})
            dataset_cfg = TEAM_CONFIG['dataset']
            assert existing.get('base_run') == run_name
            assert existing.get('seed') == seed
            assert existing.get('train_rows') == dataset_cfg['expected_rows']['train']
            assert signature.get('train_sha256') == dataset_cfg['sha256'][dataset_cfg['train_file']]
            assert signature.get('validation_sha256') == dataset_cfg['sha256'][dataset_cfg['validation_file']]
            assert previous_metadata.get('git_commit') == GIT_COMMIT, (
                'Job hoàn thành bằng commit khác với notebook hiện tại'
            )
            assert previous_metadata.get('config_sha256') == CONFIG_SHA256, (
                'Job hoàn thành bằng config khác với notebook hiện tại'
            )
            previous_metadata['status'] = 'complete'
            previous_metadata['run_summary_sha256'] = sha256_file(summary_path)
            write_json(metadata_path, previous_metadata)
            if protocol['delete_last_checkpoint_after_success'] and last_path.exists():
                last_path.unlink()
            print('Đã hoàn thành và đúng protocol, bỏ qua.')
            continue

        if resume:
            assert previous_metadata, 'Có last.pt nhưng thiếu team_job_metadata.json'
            assert previous_metadata.get('git_commit') == GIT_COMMIT, (
                'Không resume checkpoint bằng commit code khác'
            )
            assert previous_metadata.get('config_sha256') == CONFIG_SHA256, (
                'Không resume checkpoint bằng config khác'
            )
            assert previous_metadata.get('member_id') == MEMBER_ID
            assert previous_metadata.get('run') == run_name
            assert previous_metadata.get('seed') == seed
        unexpected = [
            name for name in ('best.pt', 'train_log.jsonl', 'val_metrics.json')
            if (output_dir / name).exists()
        ]
        if unexpected and not resume:
            raise RuntimeError(
                f'{run_label} có artifact dở nhưng thiếu last.pt: {unexpected}'
            )

        command = [
            sys.executable, '-m', 'experiments.run_experiment',
            '--run', run_name,
            '--seed', str(seed),
            '--epochs', str(protocol['epochs']),
            '--batch-size', str(protocol['batch_size']),
            '--eval-batch-size', str(protocol['eval_batch_size']),
            '--grad-accum-steps', str(protocol['grad_accum_steps']),
            '--num-workers', str(protocol['num_workers']),
            '--pad-to-multiple-of', str(protocol['pad_to_multiple_of']),
            '--output-dir', str(output_dir),
        ]
        if resume:
            command.append('--resume')

        metadata = {
            'schema_version': 1,
            'experiment_name': TEAM_CONFIG['experiment_name'],
            'member_id': MEMBER_ID,
            'role': MEMBER['role'],
            'run': run_name,
            'seed': seed,
            'status': 'running',
            'resumed': resume,
            'resume_count': int(previous_metadata.get('resume_count', 0)) + int(resume),
            'started_at_utc': previous_metadata.get('started_at_utc', utc_now()),
            'git_commit': GIT_COMMIT,
            'config_sha256': CONFIG_SHA256,
            'command': command,
        }
        write_json(metadata_path, metadata)

        try:
            subprocess.run(command, cwd=PROJECT, env=os.environ.copy(), check=True)
            assert summary_path.exists(), f'Thiếu run_summary.json sau {run_label}'
            metadata['status'] = 'complete'
            metadata['completed_at_utc'] = utc_now()
            metadata['run_summary_sha256'] = sha256_file(summary_path)
            write_json(metadata_path, metadata)
            if protocol['delete_last_checkpoint_after_success'] and last_path.exists():
                size_gb = last_path.stat().st_size / (1024 ** 3)
                last_path.unlink()
                print(f'Đã xóa last.pt ({size_gb:.2f} GB) vì job đã hoàn thành.')
        except BaseException as error:
            metadata['status'] = 'interrupted_or_failed'
            metadata['updated_at_utc'] = utc_now()
            metadata['error'] = repr(error)
            write_json(metadata_path, metadata)
            raise

    print('\nToàn bộ hàng đợi của thành viên đã hoàn thành.')
else:
    print('Member 4 không chạy train. Chuyển tới phần tổng hợp bên dưới.')


## 7. Member 4 kiểm tra đủ 18 job và tính analytics

In [ ]:
import pandas as pd

if MEMBER['role'] == 'analyst':
    protocol = TEAM_CONFIG['protocol']
    trainer_members = [
        (int(member_id), details)
        for member_id, details in TEAM_CONFIG['members'].items()
        if details['role'] == 'trainer'
    ]
    expected_jobs = [
        (member_id, run_name, int(details['seed']))
        for member_id, details in trainer_members
        for run_name in protocol['runs']
    ]

    status_rows = []
    result_rows = []
    violations = []
    commits = set()
    config_hashes = set()
    dataset_cfg = TEAM_CONFIG['dataset']

    for member_id, run_name, seed in expected_jobs:
        run_label = f'{run_name}_seed{seed}'
        output_dir = RESULTS_ROOT / run_label
        summary_path = output_dir / 'run_summary.json'
        metadata_path = output_dir / 'team_job_metadata.json'
        status = 'complete' if summary_path.exists() else (
            'incomplete' if output_dir.exists() else 'missing'
        )
        status_rows.append({
            'member_id': member_id, 'run': run_name, 'seed': seed, 'status': status
        })
        if status != 'complete':
            continue

        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        metadata = (
            json.loads(metadata_path.read_text(encoding='utf-8'))
            if metadata_path.exists() else {}
        )
        runtime = summary.get('runtime', {})
        signature = summary.get('data_signature', {})
        checks = {
            'base_run': summary.get('base_run') == run_name,
            'seed': summary.get('seed') == seed,
            'full_data': not summary.get('is_debug_subset', True),
            'train_rows': summary.get('train_rows') == dataset_cfg['expected_rows']['train'],
            'train_hash': signature.get('train_sha256') == dataset_cfg['sha256'][dataset_cfg['train_file']],
            'validation_hash': signature.get('validation_sha256') == dataset_cfg['sha256'][dataset_cfg['validation_file']],
            'epochs': summary.get('epochs') == protocol['epochs'],
            'batch_size': runtime.get('global_batch_size') == protocol['batch_size'],
            'eval_batch_size': runtime.get('global_eval_batch_size') == protocol['eval_batch_size'],
            'world_size': runtime.get('world_size') == 1,
            'deterministic': runtime.get('deterministic') is True,
            'amp': runtime.get('amp') is True,
            't4_gpu': any(
                'T4' in str(device).upper() for device in runtime.get('devices', [])
            ),
            'padding': runtime.get('pad_to_multiple_of') == protocol['pad_to_multiple_of'],
            'metadata_complete': metadata.get('status') == 'complete',
            'assigned_member': metadata.get('member_id') == member_id,
            'config_hash': metadata.get('config_sha256') == CONFIG_SHA256,
            'git_commit_present': bool(metadata.get('git_commit')),
        }
        failed_checks = [name for name, passed in checks.items() if not passed]
        if failed_checks:
            violations.append({'job': run_label, 'failed_checks': failed_checks})
        if metadata.get('git_commit'):
            commits.add(metadata['git_commit'])
        if metadata.get('config_sha256'):
            config_hashes.add(metadata['config_sha256'])

        result_rows.append({
            'member_id': member_id,
            'run': run_name,
            'seed': seed,
            'best_epoch': summary['best_epoch'],
            'val_f1_macro': summary['best_val_f1_macro'],
            'val_accuracy': summary['best_val_accuracy'],
            'val_loss': summary['best_val_loss'],
            'train_hours': summary['total_train_seconds'] / 3600,
            'peak_vram_gb': summary['peak_vram_gb'],
            'git_commit': metadata.get('git_commit', 'MISSING'),
            'protocol_ok': not failed_checks,
        })

    if len(commits) > 1:
        violations.append({
            'job': '_team', 'failed_checks': ['multiple_git_commits']
        })
    if config_hashes and config_hashes != {CONFIG_SHA256}:
        violations.append({
            'job': '_team', 'failed_checks': ['mixed_or_outdated_config']
        })

    status_df = pd.DataFrame(status_rows)
    raw_df = pd.DataFrame(result_rows)
    display(status_df)
    print('Hoàn thành:', int((status_df['status'] == 'complete').sum()), '/', len(status_df))
    print('Số commit code:', len(commits), sorted(commits))
    print('Số config hash:', len(config_hashes), sorted(config_hashes))
    print('Vi phạm protocol:', len(violations))
    if violations:
        display(pd.DataFrame(violations))
else:
    print('Phần này dành cho Member 4.')


## 8. Member 4 xuất bảng, biểu đồ và báo cáo

In [ ]:
if MEMBER['role'] == 'analyst':
    ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
    status_df.to_csv(ANALYSIS_ROOT / 'job_status.csv', index=False)
    raw_df.to_csv(ANALYSIS_ROOT / 'validation_by_seed.csv', index=False)

    if raw_df.empty:
        print('Chưa có job hoàn thành để phân tích.')
    else:
        import matplotlib.pyplot as plt
        import numpy as np

        aggregate_df = (
            raw_df.groupby('run', as_index=False)
            .agg(
                completed_seeds=('seed', 'count'),
                val_f1_mean=('val_f1_macro', 'mean'),
                val_f1_std=('val_f1_macro', 'std'),
                val_accuracy_mean=('val_accuracy', 'mean'),
                val_accuracy_std=('val_accuracy', 'std'),
                train_hours_mean=('train_hours', 'mean'),
                peak_vram_gb_max=('peak_vram_gb', 'max'),
            )
            .sort_values('val_f1_mean', ascending=False)
            .reset_index(drop=True)
        )
        aggregate_df['rank'] = np.arange(1, len(aggregate_df) + 1)

        baseline = 'B2_bert_finetuned_cls'
        pivot = raw_df.pivot(index='seed', columns='run', values='val_f1_macro')
        delta_rows = []
        if baseline in pivot.columns:
            for run_name in protocol['runs']:
                if run_name == baseline or run_name not in pivot.columns:
                    continue
                paired = (pivot[run_name] - pivot[baseline]).dropna()
                for seed, delta in paired.items():
                    delta_rows.append({
                        'run': run_name, 'seed': int(seed),
                        'delta_f1_vs_B2': float(delta),
                    })
        delta_df = pd.DataFrame(delta_rows)
        if not delta_df.empty:
            delta_summary_df = (
                delta_df.groupby('run', as_index=False)
                .agg(
                    paired_seeds=('seed', 'count'),
                    delta_f1_mean=('delta_f1_vs_B2', 'mean'),
                    delta_f1_std=('delta_f1_vs_B2', 'std'),
                )
                .sort_values('delta_f1_mean', ascending=False)
            )
        else:
            delta_summary_df = pd.DataFrame(
                columns=['run', 'paired_seeds', 'delta_f1_mean', 'delta_f1_std']
            )

        aggregate_df.to_csv(ANALYSIS_ROOT / 'validation_aggregate.csv', index=False)
        delta_df.to_csv(ANALYSIS_ROOT / 'paired_deltas_vs_B2.csv', index=False)
        delta_summary_df.to_csv(ANALYSIS_ROOT / 'paired_delta_summary.csv', index=False)
        write_json(ANALYSIS_ROOT / 'protocol_violations.json', violations)

        plot_df = aggregate_df.sort_values('val_f1_mean')
        figure, axis = plt.subplots(figsize=(9, 4.8))
        errors = plot_df['val_f1_std'].fillna(0)
        axis.barh(plot_df['run'], plot_df['val_f1_mean'], xerr=errors, color='#287271')
        axis.set_xlabel('Validation macro F1 trung bình')
        axis.set_title('So sánh model qua ba seed')
        axis.grid(axis='x', alpha=0.25)
        figure.tight_layout()
        figure.savefig(ANALYSIS_ROOT / 'validation_f1_mean_std.png', dpi=180)
        plt.show()

        def markdown_table(frame):
            view = frame.copy()
            for column in view.select_dtypes(include='number').columns:
                view[column] = view[column].map(
                    lambda value: '' if pd.isna(value) else f'{value:.6f}'
                )
            headers = [str(column) for column in view.columns]
            lines = ['| ' + ' | '.join(headers) + ' |']
            lines.append('| ' + ' | '.join(['---'] * len(headers)) + ' |')
            for row in view.astype(str).itertuples(index=False, name=None):
                lines.append('| ' + ' | '.join(row) + ' |')
            return '\n'.join(lines)

        complete_count = int((status_df['status'] == 'complete').sum())
        experiment_name = TEAM_CONFIG['experiment_name']
        report = [
            '# Báo cáo validation ba seed',
            '',
            f'- Thí nghiệm: `{TEAM_CONFIG["experiment_name"]}`',
            f'- Job hoàn thành: **{complete_count}/{len(status_df)}**',
            f'- Số commit code xuất hiện: **{len(commits)}**',
            f'- Số vi phạm protocol: **{len(violations)}**',
            '- Tập test: **chưa sử dụng**',
            '',
            '## Kết quả tổng hợp',
            '',
            markdown_table(aggregate_df),
            '',
            '## Chênh lệch Macro F1 so với B2',
            '',
            markdown_table(delta_summary_df),
            '',
            'Kết quả trên mới là validation. Không dùng bảng này như kết quả test cuối cùng.',
        ]
        (ANALYSIS_ROOT / 'ANALYSIS_REPORT.md').write_text(
            '\n'.join(report) + '\n', encoding='utf-8'
        )

        display(aggregate_df)
        display(delta_summary_df)
        print('Đã xuất báo cáo tại:', ANALYSIS_ROOT)
else:
    print('Phần xuất analytics dành cho Member 4.')


## 9. Sau khi Member 4 tổng hợp

Chỉ coi giai đoạn này hoàn thành khi đủ `18/18` job, chỉ có một commit code, một config hash và không có vi phạm protocol. Nhóm chọn kiến trúc bằng validation, chốt mọi quyết định, rồi giao một người duy nhất thực hiện quy trình final test được mô tả trong `HUONG_DAN_TRAIN_DATA_MOI.md`.

Member 4 **không chọn seed tốt nhất**. Cần báo cáo trung bình và độ lệch chuẩn của cả ba seed.
